# 01 — Extração e triagem de escopo

**Entrada:** `AKCIT/MedPT` no HuggingFace (384k+ Q&A de interações reais paciente–médico).
**Saída:** `../data/pares_avaliados.jsonl` (todos os pares julgados, com justificativa) e `../data/corpus_escopo.parquet`.

Este notebook faz chamadas pagas de API. Ele grava cache em disco e não deve ser
re-executado para ajustar decisões das etapas seguintes.

In [ ]:
from pathlib import Path
import json

import duckdb
import pandas as pd

DATA = Path.cwd().parent / "data"
DATA.mkdir(exist_ok=True)
PARES_AVALIADOS = DATA / "pares_avaliados.jsonl"
PARES_LEGACY = DATA / "pares_selecionados.legacy.json"
CORPUS_ESCOPO = DATA / "corpus_escopo.parquet"

SEED = 42
con = duckdb.connect()

Vamos usar o **MedPT**, dataset construído com interações reais entre pacientes e médicos.
A lib `duckdb` permite consultar os parquets direto no HuggingFace, sem baixar tudo.

In [ ]:
medpt = con.sql("""
     SELECT *
       FROM 'hf://datasets/AKCIT/MedPT/**/*.parquet'
""").df()

print(f"{len(medpt):,} linhas | colunas: {list(medpt.columns)}")

As colunas `question` e `answer` são as que alimentam o fine-tuning. Mas 384k+ perguntas
é grande demais para o escopo do exercício, então precisamos filtrar.

Seguindo a temática dos dois primeiros Tech Challenges, tratamos de **saúde da mulher**,
simulando um **hospital maternidade**: avaliação pré e neonatal. As colunas
`medical_specialty`, `condition` e `question_type` permitem chegar a um corpus viável
por filtragem estruturada.

In [ ]:
speciality_ranking = con.sql("""
     SELECT trim(esp) AS especialidade,
            COUNT(*) AS n
       FROM medpt,
            UNNEST(string_split(medical_specialty, ',')) AS t(esp)
      WHERE medical_specialty ILIKE '%Pediatra%'
         OR medical_specialty ILIKE '%Ginecologista%'
      GROUP BY 1
      ORDER BY 2 DESC
""").df()

speciality_ranking.head(10)

In [ ]:
speciality_ranking["acum"] = (
    speciality_ranking["n"].cumsum() / speciality_ranking["n"].sum()
)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
speciality_ranking["acum"].plot(ax=ax, marker="o", markersize=3)
ax.axhline(0.8, ls="--", lw=1)
ax.axhline(0.9, ls="--", lw=1)
ax.set_xticks(range(len(speciality_ranking)))
ax.set_xticklabels(speciality_ranking["especialidade"], rotation=90, fontsize=7)
ax.set_ylabel("Cobertura acumulada")
plt.tight_layout()

In [ ]:
conditions_mspecialtys = con.sql("""
    SELECT condition,
           medical_specialty,
           COUNT(*) AS n
      FROM medpt
     WHERE medical_specialty ILIKE '%Pediatra%'
        OR medical_specialty ILIKE '%Ginecologista%'
     GROUP BY 1, 2
     ORDER BY n DESC
""").df()

pairs = list(zip(conditions_mspecialtys["condition"],
                 conditions_mspecialtys["medical_specialty"]))

print(f"{len(pairs):,} pares (condition, medical_specialty) a julgar")
conditions_mspecialtys.head(10)

## Decisão — por que o juiz recebe exemplos de Q&A

A primeira versão deste notebook julgava o escopo a partir **apenas do rótulo**:
`("Constipação em bebês", "Pediatra")`. Para `Pré-eclâmpsia` o rótulo basta. Para
`Febre em crianças` ou `Coronavírus COVID-19` não — essas condições contêm pergunta
de recém-nascido e de criança de 6 anos no mesmo balde, e o rótulo não distingue o
que ele próprio não separa.

Passamos a amostrar `K_EXEMPLOS` perguntas reais de cada par e mostrá-las ao juiz.
Mesmo número de chamadas, mais tokens por chamada, veredito mais informado.

**Limite conhecido:** se a condição for internamente heterogênea, nenhum `K` resolve —
um veredito por par continua forçando um sim/não sobre um balde misto. O diff contra
o `legacy`, mais adiante, mostra quantas condições mudaram de lado; se muitas mudarem
*e* forem justamente as ambíguas, o próximo passo é julgamento linha a linha.

**Botões desta etapa:** `K_EXEMPLOS`, `TRUNCA_RESPOSTA` e a redação do `SYSTEM`.

In [ ]:
K_EXEMPLOS = 5          # quantas perguntas reais mostrar ao juiz por par
TRUNCA_RESPOSTA = 300   # chars de answer por exemplo (0 = nao mostrar resposta)

alvo = set(pairs)
exemplos_por_par = {
    chave: grupo.sample(min(K_EXEMPLOS, len(grupo)), random_state=SEED)
    for chave, grupo in medpt.groupby(["condition", "medical_specialty"])
    if chave in alvo
}

def formata_exemplos(par):
    grupo = exemplos_por_par.get(par)
    if grupo is None or len(grupo) == 0:
        return "(sem exemplos disponiveis)"
    linhas = []
    for i, row in enumerate(grupo.itertuples(), 1):
        linhas.append(f"{i}. P: {str(row.question).strip()}")
        if TRUNCA_RESPOSTA:
            linhas.append(f"   R: {str(row.answer).strip()[:TRUNCA_RESPOSTA]}")
    return "\n".join(linhas)

print(formata_exemplos(pairs[0]))

In [ ]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
if not os.getenv("Z_API_KEY"):
    raise RuntimeError("Z_API_KEY ausente. Defina no .env (veja .env.example).")

client = AsyncOpenAI(
    api_key=os.getenv("Z_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/",
)

O schema abaixo declara `required` e `additionalProperties: False`, e as chaves são
exatamente as que o código lê adiante. Na versão anterior o schema era um mapa livre
sem `required`, e o código lia uma chave que o schema não exigia — funcionava só
porque os exemplos do prompt induziam a chave, e quebraria em silêncio se o modelo
variasse.

O campo `motivo` existe para que a curadoria seja auditável: sem ele não há como
responder "por que essa condição ficou de fora?" nem depurar o juiz.

In [ ]:
SCHEMA_ESCOPO = {
    "type": "object",
    "properties": {
        "escopo": {
            "type": "integer",
            "enum": [0, 1],
            "description": "1 se a condicao pertence ao universo da maternidade, 0 caso contrario",
        },
        "motivo": {
            "type": "string",
            "description": "justificativa curta, em uma frase",
        },
    },
    "required": ["escopo", "motivo"],
    "additionalProperties": False,
}

SYSTEM = (
    "Voce e um medico que atua em um hospital maternidade, acompanhando "
    "gestantes, puerperas e bebes ate 1 ano.\n\n"
    "Tarefa: decidir se uma condicao medica pertence ao escopo de atendimento "
    "desse hospital. Voce recebe a condicao, a especialidade e alguns exemplos "
    "reais de perguntas feitas sobre essa condicao. Os exemplos mostram como a "
    "condicao aparece na pratica; use-os para desambiguar rotulos genericos.\n\n"
    "O escopo cobre:\n"
    "- fertilidade e planejamento reprodutivo\n"
    "- gestacao e suas complicacoes\n"
    "- parto\n"
    "- puerperio e amamentacao\n"
    "- bebe ate 1 ano de idade\n\n"
    "CRITERIO. Responda 1 quando pelo menos uma for verdadeira:\n"
    "(a) a condicao so ocorre na gestacao, no parto, no puerperio ou no "
    "recem-nascido;\n"
    "(b) a condicao ocorre tambem fora desse contexto, mas o diagnostico, a "
    "conduta ou o risco mudam de forma relevante quando a paciente esta "
    "gestante ou quando o paciente e um bebe ate 1 ano.\n\n"
    "Responda 0 quando a condicao apenas pode coexistir com a gestacao ou com "
    "o primeiro ano de vida sem que isso altere o manejo, e quando ela nao "
    "pertence a nenhuma das fases acima.\n\n"
    "PAPEL DOS EXEMPLOS. Se os exemplos mostrarem que as perguntas reais sobre "
    "essa condicao tratam majoritariamente de pacientes fora do escopo (criancas "
    "maiores de 1 ano, mulheres fora do ciclo gravidico-puerperal), responda 0 "
    "mesmo que o rotulo da condicao pareca pertinente.\n\n"
    "PAPEL DA ESPECIALIDADE. A condicao decide primeiro. Use a especialidade "
    "apenas para desempatar condicoes ambiguas.\n\n"
    "Exemplos de veredito:\n"
    "Pre-eclampsia | Ginecologista -> 1\n"
    "Asfixia Neonatal | Oncologista, Pediatra -> 1\n"
    "Infeccao Urinaria | Ginecologista -> 1\n"
    "Constipacao em bebes | Pediatra -> 1\n"
    "Menopausa | Ginecologista -> 0\n"
    "Escoliose | Pediatra -> 0\n"
    "Labirintite | Ginecologista -> 0\n\n"
    "Responda apenas o JSON, sem texto antes ou depois."
)

`classify` agora tem backoff exponencial — a versão anterior não tinha retry no juiz
(só na reescrita), então uma falha de rede perdia o par silenciosamente.

E a função devolve um **registro**, não um booleano. A versão anterior retornava `bool`
e o chamador fazia `zip`, o que descartava os rejeitados e todo o raciocínio junto.

In [ ]:
import asyncio
import random
import re

def parse_json(texto: str):
    limpo = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto.strip()).strip()
    return json.loads(limpo)

CONCORRENCIA = 50
MODELO_JUIZ = "glm-5.3-flash"
sem = asyncio.Semaphore(CONCORRENCIA)

async def classify(par, tentativas=4):
    cond, esp = par
    user = (
        f"Condicao: {cond} | Especialidade: {esp}\n\n"
        f"Exemplos reais de perguntas sobre essa condicao:\n{formata_exemplos(par)}"
    )
    base = {
        "condition": cond,
        "medical_specialty": esp,
        "n_exemplos": len(exemplos_por_par.get(par, [])),
    }
    async with sem:
        for tentativa in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model=MODELO_JUIZ,
                    messages=[
                        {"role": "system", "content": SYSTEM},
                        {"role": "user", "content": user},
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {"name": "classificacao", "schema": SCHEMA_ESCOPO},
                    },
                    temperature=0,
                )
                r = parse_json(resp.choices[0].message.content)
                return base | {"escopo": int(r["escopo"]), "motivo": r.get("motivo", "")}
            except Exception as erro:
                if tentativa == tentativas - 1:
                    return base | {"escopo": None, "motivo": f"ERRO: {type(erro).__name__}: {erro}"}
                await asyncio.sleep(2 ** tentativa + random.random())

In [ ]:
# Cache em disco: a triagem custa dinheiro e nao deve rodar de novo sem intencao.
if PARES_AVALIADOS.exists():
    avaliados = [json.loads(l) for l in PARES_AVALIADOS.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"Cache carregado: {len(avaliados)} pares. Apague {PARES_AVALIADOS.name} para re-julgar.")
else:
    avaliados = await asyncio.gather(*(classify(p) for p in pairs))
    with PARES_AVALIADOS.open("w", encoding="utf-8") as f:
        for reg in avaliados:
            f.write(json.dumps(reg, ensure_ascii=False) + "\n")
    print(f"{len(avaliados)} pares julgados e gravados em {PARES_AVALIADOS.name}")

In [ ]:
avaliados_df = pd.DataFrame(avaliados)

aprovados_df = avaliados_df[avaliados_df["escopo"] == 1]
rejeitados_df = avaliados_df[avaliados_df["escopo"] == 0]

print(f"{len(avaliados_df):,} pares julgados")
print(f"  aprovados:  {len(aprovados_df):,}")
print(f"  rejeitados: {len(rejeitados_df):,}")
print(f"  erros:      {avaliados_df['escopo'].isna().sum():,}")

# Amostra das justificativas dos dois lados -- e o que torna a curadoria auditavel.
for rotulo, subset in [("APROVADOS", aprovados_df), ("REJEITADOS", rejeitados_df)]:
    print(f"\n=== {rotulo} ===")
    for r in subset.sample(min(5, len(subset)), random_state=SEED).itertuples():
        print(f"- {r.condition} ({r.medical_specialty}): {r.motivo}")

## Diff contra o juiz anterior

O `legacy` é o resultado da triagem que via apenas o rótulo. Comparar os dois mede o
efeito dos exemplos — e é material direto para a seção de avaliação do relatório.

O que olhar: se as condições que mudaram de lado forem justamente as ambíguas
(`Febre em crianças`, `Coronavírus COVID-19`), os exemplos fizeram o trabalho esperado.
Se condições inequívocas mudaram, o prompt regrediu.

In [ ]:
if PARES_LEGACY.exists():
    legacy = {tuple(x) for x in json.loads(PARES_LEGACY.read_text(encoding="utf-8"))}
    novo = set(zip(aprovados_df["condition"], aprovados_df["medical_specialty"]))

    entraram, sairam = novo - legacy, legacy - novo
    print(f"legacy: {len(legacy)} aprovados | novo: {len(novo)} aprovados")
    print(f"entraram: {len(entraram)} | sairam: {len(sairam)} | estaveis: {len(novo & legacy)}\n")

    motivos = avaliados_df.set_index(["condition", "medical_specialty"])["motivo"].to_dict()
    for rotulo, conjunto in [("ENTRARAM", entraram), ("SAIRAM", sairam)]:
        print(f"=== {rotulo} ===")
        for par in list(conjunto)[:15]:
            print(f"- {par[0]} ({par[1]}): {motivos.get(par, '')}")
        print()
else:
    print(f"{PARES_LEGACY.name} ausente — sem diff.")

O corpus aprovado segue para `02_curadoria.ipynb`, que aplica os filtros de qualidade,
a anonimização e a política de retenção de duplicatas.

In [ ]:
corpus = medpt.merge(
    aprovados_df[["condition", "medical_specialty"]],
    on=["condition", "medical_specialty"],
    how="inner",
)

corpus.to_parquet(CORPUS_ESCOPO, index=False)
print(f"{len(corpus):,} exemplos em {corpus['condition'].nunique()} condições "
      f"e {corpus['medical_specialty'].nunique()} combinações de especialidade")
print(f"→ {CORPUS_ESCOPO.name}")